# Medallion Architecture on Databricks - Retail Sales

- **Bronze:** raw CSV data (loaded from a Unity Catalog volume) stored as Delta tables
- **Silver:** cleansed, standardized, validated, and enriched Delta tables
- **Gold:** business-ready sales metrics and summaries

**Dataset:** customers, products, orders, and order items.

> Run the notebook from top to bottom on Databricks. The setup cell creates a catalog named `sales_demo` and three schemas: `bronze`, `silver`, and `gold`.

## 0. Prerequisites

You need a Databricks free account.

The raw source data ships as external CSV files in the `data/` folder next to this notebook:

- `data/customers.csv`
- `data/products.csv`
- `data/orders.csv`
- `data/order_items.csv`

Before running section 1, upload these four files to the Unity Catalog volume created by the setup cell (`/Volumes/sales_demo/bronze/raw_files`) using **Catalog Explorer → Upload to this volume**.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

CATALOG = "sales_demo"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_files"

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS sales_demo;
CREATE SCHEMA IF NOT EXISTS sales_demo.bronze;
CREATE SCHEMA IF NOT EXISTS sales_demo.silver;
CREATE SCHEMA IF NOT EXISTS sales_demo.gold;
CREATE VOLUME IF NOT EXISTS sales_demo.bronze.raw_files;

## 1. Load Source Data from the Volume

The raw records intentionally include common quality problems:

- Duplicate customers and order items
- Blank email and category values
- Inconsistent country and category labels
- Numeric values stored as text
- A malformed price
- An unknown customer and product
- A future order date
- Negative quantity

Each dataset is read directly from the CSV files uploaded to the `raw_files` volume.

In [ ]:
bronze_inputs = {
    name: spark.read.option("header", True).option("inferSchema", False)
        .csv(f"{VOLUME_PATH}/{name}.csv")
    for name in ("customers", "products", "orders", "order_items")
}

for name, df in bronze_inputs.items():
    display(df)

## 2. Bronze Layer: Preserve Raw Data

Bronze stores the source data with minimal change. We add ingestion metadata for traceability, then save each dataset as a managed Delta table.

In [ ]:
for name, df in bronze_inputs.items():
    bronze_df = (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source", F.lit(f"{VOLUME_PATH}/{name}.csv"))
    )
    (bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BRONZE}.{name}"))

In [ ]:
%sql
SHOW TABLES IN sales_demo.bronze;

## 3. Profile Bronze Data

Inspect row counts, schemas, nulls, duplicates, and invalid values before defining Silver rules.

In [ ]:
for table in ("customers", "products", "orders", "order_items"):
    df = spark.table(f"{BRONZE}.{table}")
    print(f"{table}: {df.count()} rows")
    df.printSchema()

print("Duplicate customer IDs:")
(spark.table(f"{BRONZE}.customers")
 .groupBy("customer_id").count()
 .filter("count > 1")
 .show())

print("Invalid product prices:")
(spark.table(f"{BRONZE}.products")
 .filter(F.col("unit_price").cast("decimal(10,2)").isNull())
 .show())

## 4. Silver Layer: Clean and Standardize Customers

Rules:

- Cast the business key and timestamp to proper types.
- Trim and format names.
- Convert blank emails to null.
- Standardize country values to `QAT`.
- Keep the latest customer record for each `customer_id`.

In [ ]:
customers_raw = spark.table(f"{BRONZE}.customers")

customers_staged = (
    customers_raw
    .select(
        F.col("customer_id").cast("int").alias("customer_id"),
        F.initcap(F.trim("customer_name")).alias("customer_name"),
        F.when(F.trim("email") == "", None)
         .otherwise(F.lower(F.trim("email"))).alias("email"),
        F.upper(F.trim("country")).alias("country_raw"),
        F.to_timestamp("updated_at").alias("updated_at"),
        "_ingested_at"
    )
    .withColumn(
        "country_code",
        F.when(F.col("country_raw").isin("QATAR", "QA", "QAT"), "QAT")
         .otherwise(F.col("country_raw"))
    )
)

latest_customer = Window.partitionBy("customer_id").orderBy(
    F.col("updated_at").desc_nulls_last(), F.col("_ingested_at").desc()
)

customers_silver = (
    customers_staged
    .withColumn("rn", F.row_number().over(latest_customer))
    .filter("rn = 1 AND customer_id IS NOT NULL")
    .drop("rn", "country_raw")
)

(customers_silver.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.customers"))
display(customers_silver)

## 5. Silver Layer: Clean Products and Quarantine Invalid Rows

Rules:

- Cast identifiers and prices.
- Standardize category casing.
- Replace blank categories with `Unknown`.
- Quarantine rows with invalid IDs or prices instead of silently dropping them.

In [ ]:
products_raw = spark.table(f"{BRONZE}.products")

products_staged = (
    products_raw
    .select(
        F.col("product_id").cast("int").alias("product_id"),
        F.initcap(F.trim("product_name")).alias("product_name"),
        F.when(F.trim("category") == "", "Unknown")
         .otherwise(F.initcap(F.trim("category"))).alias("category"),
        F.col("unit_price").cast("decimal(10,2)").alias("unit_price"),
        "_ingested_at"
    )
)

valid_products = products_staged.filter(
    "product_id IS NOT NULL AND unit_price IS NOT NULL AND unit_price >= 0"
)
invalid_products = (
    products_staged
    .filter("product_id IS NULL OR unit_price IS NULL OR unit_price < 0")
    .withColumn("rejection_reason", F.lit("Invalid product ID or price"))
)

(valid_products.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.products"))
(invalid_products.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.products_quarantine"))

display(valid_products)
display(invalid_products)

## 6. Silver Layer: Validate Orders

Rules:

- Convert IDs and dates to proper types.
- Standardize order status.
- Accept only known customers.
- Reject future dates and malformed records.

In [ ]:
orders_raw = spark.table(f"{BRONZE}.orders")
valid_customer_ids = spark.table(f"{SILVER}.customers").select("customer_id")

orders_staged = (
    orders_raw
    .select(
        F.col("order_id").cast("int").alias("order_id"),
        F.col("customer_id").cast("int").alias("customer_id"),
        F.to_date("order_date").alias("order_date"),
        F.when(F.lower(F.trim("status")).isin("complete", "completed"), "Completed")
         .when(F.lower(F.trim("status")) == "pending", "Pending")
         .otherwise(F.initcap(F.trim("status"))).alias("status"),
        "_ingested_at"
    )
)

orders_checked = orders_staged.join(
    valid_customer_ids.withColumn("customer_exists", F.lit(True)),
    "customer_id", "left"
)

valid_orders = orders_checked.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("order_date").isNotNull()) &
    (F.col("order_date") <= F.current_date()) &
    (F.col("customer_exists") == True)
).drop("customer_exists")

invalid_orders = (
    orders_checked
    .filter(
        F.col("order_id").isNull() |
        F.col("order_date").isNull() |
        (F.col("order_date") > F.current_date()) |
        F.col("customer_exists").isNull()
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("customer_exists").isNull(), "Unknown customer")
         .when(F.col("order_date") > F.current_date(), "Future order date")
         .otherwise("Invalid order record")
    )
    .drop("customer_exists")
)

(valid_orders.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.orders"))
(invalid_orders.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.orders_quarantine"))

display(valid_orders)
display(invalid_orders)

## 7. Silver Layer: Validate and Enrich Order Items

Rules:

- Remove exact duplicates.
- Require a positive quantity.
- Require matching Silver orders and products.
- Add the product price and calculate `line_total`.

In [ ]:
items_raw = (
    spark.table(f"{BRONZE}.order_items")
    .select(
        F.col("order_id").cast("int").alias("order_id"),
        F.col("product_id").cast("int").alias("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        "_ingested_at"
    )
    .dropDuplicates(["order_id", "product_id", "quantity"])
)

orders_keys = spark.table(f"{SILVER}.orders").select("order_id")
product_lookup = spark.table(f"{SILVER}.products").select(
    "product_id", "product_name", "category", "unit_price"
)

items_checked = (
    items_raw
    .join(orders_keys.withColumn("order_exists", F.lit(True)), "order_id", "left")
    .join(product_lookup, "product_id", "left")
)

valid_items = (
    items_checked
    .filter(
        (F.col("quantity") > 0) &
        (F.col("order_exists") == True) &
        F.col("unit_price").isNotNull()
    )
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .drop("order_exists")
)

invalid_items = (
    items_checked
    .filter(
        (F.col("quantity") <= 0) |
        F.col("quantity").isNull() |
        F.col("order_exists").isNull() |
        F.col("unit_price").isNull()
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("quantity") <= 0, "Quantity must be positive")
         .when(F.col("order_exists").isNull(), "Unknown order")
         .when(F.col("unit_price").isNull(), "Unknown product")
         .otherwise("Invalid order item")
    )
    .drop("order_exists")
)

(valid_items.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.order_items"))
(invalid_items.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.order_items_quarantine"))

display(valid_items)
display(invalid_items)

## 8. Validate the Silver Layer

The checks below fail fast if Silver violates core quality expectations.

In [ ]:
customers = spark.table(f"{SILVER}.customers")
products = spark.table(f"{SILVER}.products")
orders = spark.table(f"{SILVER}.orders")
items = spark.table(f"{SILVER}.order_items")

assert customers.groupBy("customer_id").count().filter("count > 1").count() == 0
assert products.filter("unit_price IS NULL OR unit_price < 0").count() == 0
assert orders.filter(F.col("order_date") > F.current_date()).count() == 0
assert items.filter("quantity <= 0 OR line_total < 0").count() == 0
assert orders.join(customers, "customer_id", "left_anti").count() == 0
assert items.join(orders, "order_id", "left_anti").count() == 0
assert items.join(products, "product_id", "left_anti").count() == 0

print("All Silver quality checks passed.")

## 9. Build a Reusable Silver Sales View

Join trusted Silver tables at transaction-line grain: one row per order and product.

In [ ]:
sales_detail = (
    spark.table(f"{SILVER}.order_items").alias("i")
    .join(spark.table(f"{SILVER}.orders").alias("o"), "order_id")
    .join(spark.table(f"{SILVER}.customers").alias("c"), "customer_id")
    .select(
        "order_id", "customer_id", "customer_name", "country_code",
        "order_date", "status", "product_id", "product_name", "category",
        "quantity", "unit_price", "line_total"
    )
)

(sales_detail.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{SILVER}.sales_detail"))
display(sales_detail)

## 10. Gold Layer: Business-Ready Metrics

Create three simple Gold tables:

1. Daily sales summary
2. Product performance
3. Customer summary

In [ ]:
sales = spark.table(f"{SILVER}.sales_detail")

gold_daily_sales = (
    sales.groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("line_total"), 2).alias("total_sales")
    )
)

gold_product_performance = (
    sales.groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("orders_count"),
        F.round(F.sum("line_total"), 2).alias("revenue")
    )
)

gold_customer_summary = (
    sales.groupBy("customer_id", "customer_name", "country_code")
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.sum("quantity").alias("units_purchased"),
        F.round(F.sum("line_total"), 2).alias("total_spent")
    )
)

for name, df in {
    "daily_sales": gold_daily_sales,
    "product_performance": gold_product_performance,
    "customer_summary": gold_customer_summary,
}.items():
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD}.{name}"))

print("Gold tables created.")

In [ ]:
display(spark.table(f"{GOLD}.daily_sales").orderBy("order_date"))
display(spark.table(f"{GOLD}.product_performance").orderBy(F.desc("revenue")))
display(spark.table(f"{GOLD}.customer_summary").orderBy(F.desc("total_spent")))

## 11. Final Architecture and Tables

```text
External CSV files (Volume)
   ↓
Bronze: raw Delta tables
   ↓
Silver: clean + standardize + validate + enrich
   ↓
Gold: aggregate metrics for reporting
```

Review all created tables:

In [ ]:
%sql
SHOW TABLES IN sales_demo.bronze;

In [ ]:
%sql
SHOW TABLES IN sales_demo.silver;

In [ ]:
%sql
SHOW TABLES IN sales_demo.gold;

## 12. Import and Run on Databricks

1. Save this notebook as `retail_sales_medallion_databricks.ipynb`.
2. Sign in to the Databricks workspace.
3. Open **Workspace** and choose your user folder or a training folder.
4. Select **Import**, then upload the `.ipynb` file.
5. Open the imported notebook and run the setup cells to create the catalog, schemas, and the `raw_files` volume.
6. Upload the four CSV files from the `data/` folder to the `sales_demo.bronze.raw_files` volume via **Catalog Explorer**.
7. Select **Run all**.
8. Open **Catalog Explorer** and inspect:
   - `sales_demo.bronze`
   - `sales_demo.silver`
   - `sales_demo.gold`
9. Review the Silver quarantine tables to understand rejected records.
10. Re-run the notebook to demonstrate idempotency. The tutorial uses overwrite mode, so repeated runs produce the same logical result.

### Optional Reset

Run the following only when you want to remove the complete tutorial environment:

```sql
DROP CATALOG IF EXISTS sales_demo CASCADE;
```